In [186]:
from pulp import *
import pandas as pd

# masses in dalton
phosphorus = 30.973761998
oxygen = 31.989829239
d_ribose = 150.05282342

tail_5prime = [d_ribose, oxygen, 2*oxygen + phosphorus, oxygen]

base_masses = pd.read_csv("../masses_all.tsv", sep="\t").set_index("nucleoside", drop=False)
# dbg:
# masses = masses.loc[["A", "C", "G", "U"]]

In [187]:
def get_representatives(base_masses):
    # collapse nucleosides that have the same mass
    return base_masses.groupby("monoisotopic_mass").apply(lambda x: x.iloc[0]).set_index("nucleoside", drop=False)

base_masses_representatives = get_representatives(base_masses)
base_masses_representatives

/tmp/ipykernel_31938/2453496861.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return base_masses.groupby("monoisotopic_mass").apply(lambda x: x.iloc[0]).set_index("nucleoside", drop=False)


,nucleoside,monoisotopic_mass
nucleoside,,
C,C,243.08552
U,U,244.06954
A,A,267.09675
1A,1A,281.11240
19A,19A,282.09640
G,G,283.09167
67A,67A,295.09170
01A,01A,295.12810
019A,019A,296.11210


In [188]:
base_masses

,nucleoside,monoisotopic_mass
nucleoside,,
A,A,267.09675
G,G,283.09167
C,C,243.08552
U,U,244.06954
1A,1A,281.11240
...,...,...
10G,10G,409.15970
102G,102G,425.15470
105G,105G,538.20230


In [189]:
import random, re
from scipy.stats import norm

nucleoside_re = re.compile(r"\d*[ACGU]")

true_sequence = nucleoside_re.findall("CCUGUU")
print(true_sequence)
n_fragments = 60

# sample random fragments from true sequence
fragment_noise = norm.rvs(scale=0.5, size=n_fragments)
def random_fragment():
    l = random.randint(0, len(true_sequence) - 1)
    r = random.randint(l + 1, len(true_sequence))
    return l, r
fragments = [random_fragment() for _ in range(n_fragments)]
fragment_masses = [max(sum(base_masses.loc[b, "monoisotopic_mass"] for b in true_sequence[l:r]) + noise, 0.0) for noise, (l, r) in zip(fragment_noise, fragments)]

start_fragments = [i for i in range(n_fragments) if fragments[i][0] == 0]
end_fragments = [i for i in range(n_fragments) if fragments[i][1] == len(true_sequence)]

fragments, fragment_masses, fragment_noise

['C', 'C', 'U', 'G', 'U', 'U']


([(2, 5),
  (5, 6),
  (2, 4),
  (0, 5),
  (3, 4),
  (1, 2),
  (0, 6),
  (1, 3),
  (4, 5),
  (4, 6),
  (5, 6),
  (0, 5),
  (1, 5),
  (5, 6),
  (1, 5),
  (0, 4),
  (2, 5),
  (4, 5),
  (4, 5),
  (0, 4),
  (1, 4),
  (0, 3),
  (0, 2),
  (1, 2),
  (3, 5),
  (2, 3),
  (0, 6),
  (0, 6),
  (2, 4),
  (1, 3),
  (4, 6),
  (3, 6),
  (3, 4),
  (4, 6),
  (1, 6),
  (5, 6),
  (2, 5),
  (1, 4),
  (4, 6),
  (4, 5),
  (5, 6),
  (1, 2),
  (4, 5),
  (2, 3),
  (0, 4),
  (1, 2),
  (4, 6),
  (0, 3),
  (4, 6),
  (4, 5),
  (1, 3),
  (3, 4),
  (5, 6),
  (1, 4),
  (5, 6),
  (0, 2),
  (0, 5),
  (0, 4),
  (1, 3),
  (1, 5)],
 [np.float64(771.0659266267483),
  np.float64(244.02247736248557),
  np.float64(527.5575178162773),
  np.float64(1256.7724847727727),
  np.float64(282.89261795076953),
  np.float64(243.39873752278956),
  np.float64(1501.546952667726),
  np.float64(487.0170466204544),
  np.float64(244.32847070874357),
  np.float64(488.0776312449231),
  np.float64(244.71314100877373),
  np.float64(1257.387881863018

In [190]:
from itertools import combinations


def reduce_alphabet_by_fragments(the_base_masses):
    # TODO: the -2.0 should be informed by the variance in the measurements
    min_plausible_nucleoside_diff = the_base_masses["monoisotopic_mass"].min() - 2.0

    start_masses = pd.Series(sorted(fragment_masses[i] for i in start_fragments))
    end_masses = pd.Series(sorted(fragment_masses[i] for i in end_fragments))

    print(start_masses)
    print(end_masses)


    def mass_diffs(masses):
        diffs = set(masses[1:].values - masses[:-1].values)
        print(diffs)
        return diffs
    
    diffs = sorted(mass_diffs(start_masses) | mass_diffs(end_masses))

    def is_similar(mass_a, mass_b):
        return abs(mass_a - mass_b) < 1.0

    def explain_diffs(diffs):
        explanations = {
            diff: [
                item.nucleoside for item in the_base_masses.itertuples()
                if is_similar(diff, item.monoisotopic_mass)
            ]
            for diff in diffs
        }
        explanations.update(
            {
                diff: [
                    (item_a.nucleoside, item_b.nucleoside)
                    for item_a, item_b in combinations(the_base_masses.itertuples(), 2)
                    if is_similar(diff, item_a.monoisotopic_mass + item_b.monoisotopic_mass)
                ]
                for diff in diffs if not explanations[diff]
            }
        )
        return explanations

    def get_similar_nucleosides():
        i_diff = 0
        sorted_masses = the_base_masses.sort_values("monoisotopic_mass")
        for item in sorted_masses.itertuples():
            # move until close enough to next mass in base_masses
            # (diffs are sorted)
            # TODO the -1.0 should be informed by the variance in the measurements
            while i_diff < len(diffs) and diffs[i_diff] < item.monoisotopic_mass - 1.0:
                i_diff += 1
            j = 0
            # TODO the +1.0 should be informed by the variance in the measurements
            while i_diff + j < len(diffs) and diffs[i_diff + j] <= item.monoisotopic_mass + 1.0:
                diff_mass = diffs[i_diff + j]
                if is_similar(diff_mass, item.monoisotopic_mass):
                    yield item.nucleoside
                j += 1

    explanations = explain_diffs(diffs)

    # unexplained differences, only consider those 
    unexplained = [diff for diff, expls in explanations.items() if not expls and diff > min_plausible_nucleoside_diff]

    if unexplained:
        # at least one fragment differs by more than two nucleosides,
        # so we can't reduce the alphabet with this heuristic so far
        print("Unexplained differences:", unexplained)
        return None
    
    def get_nucleosides(explanations):
        for expls in explanations.values():
            for expl in expls:
                if isinstance(expl, tuple):
                    yield from expl
                else:
                    yield expl

    all_nucleosides = list(set(get_nucleosides(explanations)))

    print(all_nucleosides)

    return the_base_masses.loc[all_nucleosides]

In [191]:
reduced_alphabet_masses = reduce_alphabet_by_fragments(base_masses_representatives)

0      486.201365
1      487.089307
2      729.813759
3      731.269778
4     1012.866280
5     1013.127202
6     1013.357805
7     1013.933201
8     1256.772485
9     1257.039236
10    1257.387882
11    1501.163182
12    1501.546953
13    1501.670406
dtype: float64
0      243.682665
1      243.803383
2      244.022477
3      244.101709
4      244.260055
5      244.708439
6      244.713141
7      487.520667
8      487.540036
9      488.077631
10     488.717110
11     488.808837
12     488.979606
13     771.531283
14    1258.202117
15    1501.163182
16    1501.546953
17    1501.670406
dtype: float64
{np.float64(0.8879420936490305), np.float64(1.4560189454235797), np.float64(0.2609223731720931), np.float64(0.23060352585082455), np.float64(0.5753958333372111), np.float64(0.26675158688908596), np.float64(0.34864550335646527), np.float64(0.3837711657724867), np.float64(0.12345355497905075), np.float64(242.7244515424661), np.float64(242.83928351617442), np.float64(243.77529963893517), np.flo

In [192]:
def estimate_seq(base_masses):
    prob = LpProblem("RNA sequencing", LpMinimize)
    # i = 1,...,n: positions in the sequence
    # j = 1,...,m: fragments
    # b = 1,...,k: (modified) bases

    n = len(true_sequence)
    m = len(fragment_masses)
    bases = base_masses.index.values

    # x: binary variables indicating fragment j presence at position i
    x = [[LpVariable(f"x_{i},{j}", lowBound=0, upBound=1, cat=LpInteger) for j in range(m)] for i in range(n)]
    # y: binary variables indicating base at position i
    y = [{b: LpVariable(f"y_{i},{b}", lowBound=0, upBound=1, cat=LpInteger) for b in bases} for i in range(n)]
    # z: binary variables indicating product of x and y
    z = [[{b: LpVariable(f"z_{i},{j},{b}", lowBound=0, upBound=1, cat=LpInteger) for b in bases} for j in range(m)] for i in range(n)]
    # weight_diff_abs: absolute value of weight_diff
    weight_diff_abs = [LpVariable(f"weight_diff_abs_{j}", lowBound=0, cat=LpContinuous) for j in range(m)]
    # weight_diff: difference between fragment monoisotopic mass and sum of masses of bases in fragment as estimated in the MILP
    weight_diff = [fragment_masses[j] - lpSum([z[i][j][b] * base_masses.loc[b, "monoisotopic_mass"] for i in range(n) for b in bases]) for j in range(m)]

    # optimization function
    prob += lpSum([weight_diff_abs[j] for j in range(m)])

    # select one base per position
    for i in range(n):
        
        prob += lpSum([y[i][b] for b in bases]) == 1

    # fill z with the product of binary variables x and y
    for i in range(n):
        for j in range(m):
            for b in bases:
                prob += z[i][j][b] <= x[i][j]
                prob += z[i][j][b] <= y[i][b]
                prob += z[i][j][b] >= x[i][j] + y[i][b] - 1

    # ensure that fragment is aligned continuously
    # (no gaps: if x[i,j] = 1 and x[i+2,j] = 1, then x[i+1,j] = 1)
    for j in range(m):
        for i in range(n - 2):
            prob += x[i][j] + x[i + 2][j] - 1 <= x[i + 1][j]

    # ensure that start fragments are aligned at the beginning of the sequence
    for j in start_fragments:
        x[0][j].setInitialValue(1)
        x[0][j].fixValue()

    # ensure that end fragments are aligned at the end of the sequence
    for j in end_fragments:
        x[n - 1][j].setInitialValue(1)
        x[n - 1][j].fixValue()

    # constrain weight_diff_abs to be the absolute value of weight_diff
    for j in range(m):
        prob += weight_diff_abs[j] >= weight_diff[j]
        prob += weight_diff_abs[j] >= -weight_diff[j]
    
    import pulp
    gurobi = pulp.getSolver("GUROBI_CMD")
    gurobi.msg = False
    result_ = prob.solve(gurobi)

    def get_base(i):
        for b in bases:
            if y[i][b].value() == 1:
                return b
        return None

    return [get_base(i) for i in range(n)]
    

In [193]:
from collections import defaultdict
from sklearn.cluster import KMeans
import numpy as np

reduce_to_k = 6

def reduce_alphabet(masses):
    kmeans = KMeans(n_clusters=reduce_to_k, random_state=213126)
    kmeans.fit(masses["monoisotopic_mass"].values.reshape(-1, 1))
    pseudo_nucs = [f"n{k}" for k in range(reduce_to_k)]
    nuc_mapping = defaultdict(list)
    for nuc, label in zip(masses["nucleoside"], kmeans.labels_):
        nuc_mapping[pseudo_nucs[label]].append(nuc)
    return (
        pd.DataFrame({
            "nucleoside": pseudo_nucs,
            "monoisotopic_mass": kmeans.cluster_centers_[:, 0]
        }).set_index("nucleoside", drop=False),
        nuc_mapping
    )

In [194]:
def expand_alphabet(seq, nuc_mapping):
    return {nuc for pseudo_nuc in seq for nuc in nuc_mapping[pseudo_nuc]}

In [195]:
start_fragments, end_fragments

([3, 6, 11, 15, 19, 21, 22, 26, 27, 44, 47, 55, 56, 57],
 [1, 6, 9, 10, 13, 26, 27, 30, 31, 33, 34, 35, 38, 40, 46, 48, 52, 54])

In [196]:
current_masses = base_masses_representatives
seq = None
last_masses_count = None

while True:
    print(f"remaining masses: {current_masses}")
    
    if current_masses.shape[0] > reduce_to_k and current_masses.shape[0] != last_masses_count:
        pseudo_masses, nuc_mapping = reduce_alphabet(current_masses)
        seq = estimate_seq(pseudo_masses)
        print(seq, nuc_mapping)
    else:
        seq = estimate_seq(current_masses)
        break

    last_masses_count = current_masses.shape[0]
    current_masses = current_masses.loc[list(expand_alphabet(seq, nuc_mapping))]
seq

/home/johannes/micromamba/envs/pystats/lib/python3.12/site-packages/pulp/pulp.py:1316: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


remaining masses:            nucleoside  monoisotopic_mass
nucleoside                              
G                   G          283.09167
19A               19A          282.09640
1A                 1A          281.11240
C                   C          243.08552
U                   U          244.06954


['C', 'G', 'U', 'C', 'U', 'U']